# Weighted Least Squares (WLS)

## Overview
Ordinary Least Squares assumes errors are homoscedastic (equal variance). When variance is unequal — heteroscedasticity — WLS assigns each observation a weight inversely proportional to its variance, giving less influential points to noisy observations and restoring efficiency.

**When WLS is appropriate:**
- Known variance structure (survey sampling with known sampling fractions)
- Variance proportional to a predictor or the mean (common in ecology)
- Aggregated data where observations represent different numbers of replicates
- Replicated measurements with known measurement uncertainty

---

In [ ]:
library(tidyverse); library(sandwich); library(lmtest)
set.seed(33)
# Ecological monitoring: species richness across sites of different sizes
# Larger sites surveyed more thoroughly → less sampling variance
n <- 50
sites <- data.frame(
  site_area    = runif(n, 1, 50),   # hectares
  survey_effort= round(runif(n, 5, 40)),  # person-hours
  richness     = NA
)
# True relationship: richness ~ area; variance decreases with effort
sites$richness <- with(sites,
  5 + 0.8*site_area + rnorm(n, 0, 12/sqrt(survey_effort))
)
cat("Survey dataset: n =", n, "sites\n")
cat("Variance in richness inversely proportional to survey effort\n")

In [ ]:
# Fit OLS vs WLS
m_ols <- lm(richness ~ site_area, data=sites)
# Weights = survey effort (inverse variance weights when var ∝ 1/effort)
m_wls <- lm(richness ~ site_area, data=sites, weights=survey_effort)

cat("OLS coefficients:\n"); print(coef(m_ols))
cat("\nWLS coefficients (weighted by survey effort):\n"); print(coef(m_wls))

# Visualise heteroscedasticity
par(mfrow=c(1,2))
plot(fitted(m_ols), residuals(m_ols),
     main="OLS residuals — heteroscedastic pattern",
     xlab="Fitted", ylab="Residuals"); abline(h=0,lty=2)
plot(fitted(m_wls), weighted.residuals(m_wls),
     main="WLS weighted residuals — variance stabilised",
     xlab="Fitted", ylab="Weighted residuals"); abline(h=0,lty=2)
par(mfrow=c(1,1))

In [ ]:
# Alternative: HC (heteroscedasticity-consistent) standard errors
# When you can't specify a weight structure, use robust SEs
coeftest(m_ols, vcov = sandwich::vcovHC(m_ols, type="HC3"))
cat("\nHC3 robust standard errors — valid under heteroscedasticity\n")
cat("without requiring a specific weight structure\n\n")

# When weights are unknown: iteratively reweighted LS (IRLS via GLM)
# Poisson GLM is equivalent to WLS with variance proportional to mean
m_pois <- glm(round(richness) ~ site_area, data=sites, family=poisson)
cat("Poisson GLM (IRLS): comparable when counts, variance ~ mean\n")
print(coef(m_pois))

cat("\nChoosing between WLS approaches:\n")
cat("  Known weights (effort, pop size): use lm(..., weights=w)\n")
cat("  Unknown variance structure:       use HC robust SEs\n")
cat("  Count/rate data:                  use Poisson/NB GLM\n")

---
## Common Pitfalls

**1. Ignoring heteroscedasticity and using OLS**
OLS with heteroscedastic errors produces unbiased estimates but inefficient standard errors. The SEs will be wrong — typically underestimated for high-variance observations — leading to anti-conservative tests. Breusch-Pagan test (`lmtest::bptest()`) can formally detect heteroscedasticity.

**2. Specifying weights as the variance rather than the inverse variance**
WLS weights should be proportional to `1/variance` — high-variance observations get low weight. Passing raw variances as weights (without inversion) has the opposite effect: downweighting precise observations and upweighting noisy ones.

**3. Using WLS when GLMs are more appropriate**
For count data where variance scales with the mean (Poisson variance structure), a Poisson GLM is preferable to WLS on counts. GLMs model the mean-variance relationship explicitly and handle non-negative outcomes correctly.

**4. Not checking that weights are measured independently of the response**
Weights must not be derived from the response variable itself (circular), and should not be strongly correlated with the treatment effects. Weights derived from a preliminary estimate of variance from the same dataset require iterative reweighting (IRLS).

**5. Forgetting that WLS changes the residual diagnostic plots**
Standard residual plots from `plot(lm_wls)` show raw residuals. For WLS, inspect *weighted* residuals (`weighted.residuals()`) to assess homoscedasticity after weighting.

**6. Applying WLS to aggregated data without accounting for group size**
When each row is an aggregate (mean of ni observations), weight by ni. Failing to do so treats a mean of 100 observations identically to a mean of 5, severely underusing the information in large groups.


---
*r_methods_library - Samantha McGarrigle*